# Lab 5: Building an LLM-powered Chatbot with HuggingFace and Gradio

## Introduction

In this lab, you'll learn how to create a functional chatbot powered by a Large Language Model (LLM) from HuggingFace's Transformers library. We'll implement conversation history management and build an interactive user interface using Gradio.

By the end of this lab, you'll understand:
- How to load and use pre-trained language models from HuggingFace
- How to manage conversation history for contextual responses
- How to create an interactive UI for your chatbot
- Basic techniques for prompt engineering and response generation
- How to deploy your chatbot for others to use

Let's get started!

## 1. Setup and Installation

First, let's install the necessary libraries:

In [ ]:
# Install required packages
!pip install transformers accelerate bitsandbytes sentencepiece gradio

Next, let's import the required libraries:

In [ ]:
import os
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import gradio as gr
import time
import json
from datetime import datetime

## 2. Loading a Pre-trained Language Model

For this lab, we'll use a smaller but capable LLM that can run on consumer hardware. TinyLlama is a good choice as it's a distilled version that maintains reasonable performance while being computationally efficient.

In [ ]:
# Define model name
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load the model (with quantization for efficiency)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16,
    load_in_8bit=True,  # Enable 8-bit quantization
)

print(f"Model {model_name} loaded successfully!")

## 3. Conversation History Management

Now, let's create a class to manage conversation history:

In [ ]:
class ConversationManager:
    def __init__(self, max_history=10):
        self.conversations = {}
        self.max_history = max_history
    
    def create_conversation(self, conversation_id=None):
        """Create a new conversation or reset an existing one"""
        if conversation_id is None:
            conversation_id = str(time.time())
        
        self.conversations[conversation_id] = {
            "messages": [],
            "created_at": datetime.now().isoformat(),
            "updated_at": datetime.now().isoformat()
        }
        return conversation_id
    
    def add_message(self, conversation_id, role, content):
        """Add a message to the conversation history"""
        if conversation_id not in self.conversations:
            conversation_id = self.create_conversation(conversation_id)
        
        self.conversations[conversation_id]["messages"].append({
            "role": role,
            "content": content,
            "timestamp": datetime.now().isoformat()
        })
        
        # Trim history if it exceeds max_history
        if len(self.conversations[conversation_id]["messages"]) > self.max_history * 2:
            # Keep the first message (system prompt) and the most recent messages
            self.conversations[conversation_id]["messages"] = [
                self.conversations[conversation_id]["messages"][0]
            ] + self.conversations[conversation_id]["messages"][-self.max_history * 2 + 1:]
        
        self.conversations[conversation_id]["updated_at"] = datetime.now().isoformat()
        return conversation_id
    
    def get_history(self, conversation_id):
        """Get the full conversation history"""
        if conversation_id not in self.conversations:
            return []
        return self.conversations[conversation_id]["messages"]
    
    def format_prompt(self, conversation_id, system_prompt=None):
        """Format the conversation history into a prompt for the model"""
        if conversation_id not in self.conversations:
            conversation_id = self.create_conversation(conversation_id)
        
        # Initialize with system prompt if provided and no messages exist
        if system_prompt and not self.conversations[conversation_id]["messages"]:
            self.add_message(conversation_id, "system", system_prompt)
        
        messages = self.get_history(conversation_id)
        
        # Format the conversation into a prompt string
        prompt = ""
        for message in messages:
            role = message["role"]
            content = message["content"]
            
            if role == "system":
                prompt += f"<|system|>\n{content}\n"
            elif role == "user":
                prompt += f"<|user|>\n{content}\n"
            elif role == "assistant":
                prompt += f"<|assistant|>\n{content}\n"
        
        # Add the final assistant prompt
        prompt += "<|assistant|>\n"
        
        return prompt
    
    def save_conversations(self, file_path):
        """Save all conversations to a JSON file"""
        with open(file_path, 'w') as f:
            json.dump(self.conversations, f, indent=2)
    
    def load_conversations(self, file_path):
        """Load conversations from a JSON file"""
        if os.path.exists(file_path):
            with open(file_path, 'r') as f:
                self.conversations = json.load(f)

## 4. Creating the Chatbot

Now, let's create our chatbot class that uses the model and conversation manager:

In [ ]:
class LLMChatbot:
    def __init__(self, model, tokenizer, conversation_manager=None):
        self.model = model
        self.tokenizer = tokenizer
        self.conversation_manager = conversation_manager or ConversationManager()
        
        # Set default system prompt
        self.default_system_prompt = """You are a helpful, respectful and honest assistant. 
Always answer as helpfully as possible, while being safe.
Your answers should be detailed and comprehensive.
If a question is unclear or lacks details, ask for clarification.
If you're unsure, admit it and don't share false information."""
    
    def generate_response(self, user_input, conversation_id=None, system_prompt=None, max_new_tokens=512):
        """Generate a response to the user input"""
        # Create or get conversation
        if conversation_id is None or conversation_id not in self.conversation_manager.conversations:
            conversation_id = self.conversation_manager.create_conversation()
            system_prompt = system_prompt or self.default_system_prompt
            self.conversation_manager.add_message(conversation_id, "system", system_prompt)
        
        # Add user message to history
        self.conversation_manager.add_message(conversation_id, "user", user_input)
        
        # Format the full prompt with history
        prompt = self.conversation_manager.format_prompt(conversation_id)
        
        # Generate response
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                repetition_penalty=1.1,
                pad_token_id=self.tokenizer.eos_token_id
            )
        
        # Decode the response
        full_response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        # Extract just the assistant's response
        response = full_response.split("<|assistant|>\n")[-1].strip()
        
        # Add assistant response to history
        self.conversation_manager.add_message(conversation_id, "assistant", response)
        
        return response, conversation_id
    
    def reset_conversation(self, conversation_id=None):
        """Reset or create a new conversation"""
        return self.conversation_manager.create_conversation(conversation_id)
    
    def save_conversations(self, file_path):
        """Save conversations to disk"""
        self.conversation_manager.save_conversations(file_path)
    
    def load_conversations(self, file_path):
        """Load conversations from disk"""
        self.conversation_manager.load_conversations(file_path)

## 5. Building the Gradio UI

Now, let's create a user interface using Gradio:

In [ ]:
def create_chatbot_ui(chatbot):
    """Create a Gradio UI for the chatbot"""
    
    # Initialize conversation ID
    current_conversation_id = None
    
    def respond(message, history, conversation_id):
        """Generate response and update the UI"""
        response, new_conversation_id = chatbot.generate_response(message, conversation_id)
        return response, new_conversation_id
    
    def reset_chat():
        """Reset the conversation"""
        nonlocal current_conversation_id
        current_conversation_id = chatbot.reset_conversation()
        return [], current_conversation_id
    
    def save_chats():
        """Save conversations to file"""
        chatbot.save_conversations("conversations.json")
        return "Conversations saved!"
    
    def load_chats():
        """Load conversations from file"""
        try:
            chatbot.load_conversations("conversations.json")
            return "Conversations loaded!"
        except:
            return "No saved conversations found."
    
    with gr.Blocks(css="footer {visibility: hidden}") as demo:
        gr.Markdown("# 🤖 LLM Chatbot")
        gr.Markdown("This chatbot is powered by a TinyLlama model from HuggingFace.")
        
        conversation_id = gr.State(value=None)
        
        with gr.Row():
            with gr.Column(scale=4):
                chatbot_ui = gr.Chatbot(height=500, show_label=False)
                
                with gr.Row():
                    with gr.Column(scale=8):
                        msg = gr.Textbox(
                            show_label=False,
                            placeholder="Type your message here...",
                            container=False
                        )
                    with gr.Column(scale=1):
                        submit_btn = gr.Button("Send")
            
            with gr.Column(scale=1):
                clear_btn = gr.Button("🗑️ New Chat")
                save_btn = gr.Button("💾 Save Chats")
                load_btn = gr.Button("📂 Load Chats")
                status = gr.Textbox(label="Status", interactive=False)
        
        # Set up event handlers
        msg.submit(
            respond,
            [msg, chatbot_ui, conversation_id],
            [chatbot_ui, conversation_id],
            api_name="chat"
        ).then(lambda: "", None, msg)
        
        submit_btn.click(
            respond,
            [msg, chatbot_ui, conversation_id],
            [chatbot_ui, conversation_id]
        ).then(lambda: "", None, msg)
        
        clear_btn.click(reset_chat, None, [chatbot_ui, conversation_id])
        save_btn.click(save_chats, None, status)
        load_btn.click(load_chats, None, status)
        
        # Initialize conversation
        demo.load(reset_chat, None, [chatbot_ui, conversation_id])
    
    return demo

## 6. Launch the Chatbot

Finally, let's launch our chatbot:

In [ ]:
# Create conversation manager and chatbot
conversation_manager = ConversationManager()
chatbot = LLMChatbot(model, tokenizer, conversation_manager)

# Create and launch the UI
demo = create_chatbot_ui(chatbot)
demo.launch(share=True)  # set share=False if you don't want to create a public link

## 7. Advanced Extensions (Optional)

Here are some advanced features you can add to your chatbot:

### 7.1 Adding Memory Management

In [ ]:
# Optional: Install additional libraries for vector storage
# !pip install sentence-transformers faiss-cpu

In [ ]:
class EnhancedConversationManager(ConversationManager):
    def __init__(self, max_history=10, use_vector_db=False):
        super().__init__(max_history)
        self.use_vector_db = use_vector_db
        
        if use_vector_db:
            # Import additional libraries for vector database
            from sentence_transformers import SentenceTransformer
            import faiss
            
            # Initialize embedding model and vector database
            self.embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
            self.vector_dbs = {}
    
    def add_message_with_embedding(self, conversation_id, role, content):
        """Add a message and its embedding to the conversation history"""
        super().add_message(conversation_id, role, content)
        
        if self.use_vector_db:
            if conversation_id not in self.vector_dbs:
                # Initialize vector DB for this conversation
                embedding_dim = self.embedding_model.get_sentence_embedding_dimension()
                self.vector_dbs[conversation_id] = faiss.IndexFlatL2(embedding_dim)
            
            # Add embedding to vector DB
            embedding = self.embedding_model.encode([content])[0]
            self.vector_dbs[conversation_id].add(embedding.reshape(1, -1))
    
    def search_similar_messages(self, conversation_id, query, k=3):
        """Search for similar messages in the conversation history"""
        if not self.use_vector_db or conversation_id not in self.vector_dbs:
            return []
        
        # Generate embedding for query
        query_embedding = self.embedding_model.encode([query])[0]
        
        # Search for similar messages
        distances, indices = self.vector_dbs[conversation_id].search(
            query_embedding.reshape(1, -1), k
        )
        
        # Get the messages
        messages = self.conversations[conversation_id]["messages"]
        return [messages[i] for i in indices[0] if i < len(messages)]

### 7.2 Adding RAG (Retrieval-Augmented Generation)

In [ ]:
class RAGChatbot(LLMChatbot):
    def __init__(self, model, tokenizer, conversation_manager=None, documents=None):
        super().__init__(model, tokenizer, conversation_manager)
        
        # Import additional libraries for RAG
        from sentence_transformers import SentenceTransformer
        import faiss
        
        # Initialize document store
        self.documents = documents or []
        self.embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
        self.index = None
        
        # Build index if documents are provided
        if self.documents:
            self.build_index()
    
    def build_index(self):
        """Build a FAISS index from the documents"""
        # Get embeddings for documents
        embeddings = self.embedding_model.encode(self.documents)
        
        # Build FAISS index
        embedding_dim = self.embedding_model.get_sentence_embedding_dimension()
        self.index = faiss.IndexFlatL2(embedding_dim)
        self.index.add(embeddings)
    
    def add_documents(self, new_documents):
        """Add new documents to the index"""
        if not new_documents:
            return
        
        self.documents.extend(new_documents)
        
        # Get embeddings for new documents
        embeddings = self.embedding_model.encode(new_documents)
        
        # Create index if it doesn't exist
        if self.index is None:
            embedding_dim = self.embedding_model.get_sentence_embedding_dimension()
            self.index = faiss.IndexFlatL2(embedding_dim)
        
        # Add to index
        self.index.add(embeddings)
    
    def retrieve_relevant_documents(self, query, k=3):
        """Retrieve relevant documents for the query"""
        if self.index is None or not self.documents:
            return []
        
        # Get embedding for query
        query_embedding = self.embedding_model.encode([query])[0]
        
        # Search for similar documents
        distances, indices = self.index.search(
            query_embedding.reshape(1, -1), k
        )
        
        # Return the relevant documents
        return [self.documents[i] for i in indices[0]]
    
    def generate_response(self, user_input, conversation_id=None, system_prompt=None, max_new_tokens=512):
        """Generate a response using RAG"""
        # Retrieve relevant documents
        relevant_docs = self.retrieve_relevant_documents(user_input)
        
        # Create context from relevant documents
        context = ""
        if relevant_docs:
            context = "Based on the following information:\n" + "\n".join(relevant_docs) + "\n\n"
        
        # Augment the user input with the context
        augmented_input = context + user_input
        
        # Generate response using the parent method
        return super().generate_response(augmented_input, conversation_id, system_prompt, max_new_tokens)

## 8. Conclusion and Additional Resources

In this lab, you've learned how to:
1. Load and use a pre-trained language model from HuggingFace
2. Implement conversation history management
3. Create an interactive UI with Gradio
4. Extend your chatbot with advanced features (optional)

This is just the beginning! You can further enhance your chatbot by:
- Fine-tuning the model on domain-specific data
- Implementing more sophisticated prompt templates
- Adding integration with external APIs
- Implementing advanced techniques like chain-of-thought prompting
- Deploying your chatbot to a production environment

### Additional Resources:
- [HuggingFace Transformers Documentation](https://huggingface.co/docs/transformers/index)
- [Gradio Documentation](https://www.gradio.app/docs/)
- [LangChain Documentation](https://python.langchain.com/docs/get_started/introduction)
- [FAISS Documentation](https://faiss.ai/index.html)

Happy building!